# import and data

In [1]:
# =============================================================================
# imports
# =============================================================================
import sqlite3
import pandas as pd
import sys
import os

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
import utils

# =============================================================================
# Load configuration
# =============================================================================
config = utils.load_db_config()
config["database"]["features"] = utils.load_features_config()["database"]["features"]
models_cfg = utils.load_models_config().get("models", {})
MODEL_ID = "lg_l_rw240_p90_base_sm"
if MODEL_ID not in models_cfg:
    raise KeyError(f"Model not found in config/models.json: {MODEL_ID}")
config["model"] = models_cfg[MODEL_ID]
config["api"] = utils.load_env_config().get("api", {})

# -------------------------------------------------------------------------
# Extract database config
# -------------------------------------------------------------------------
DB_PATH       = config["database"]["db_path"]
tables_cfg    = config["database"]["tables"]
features_cfg  = config["database"]["features"]

# -------------------------------------------------------------------------
# Extract table names
# -------------------------------------------------------------------------
table_features    = tables_cfg["features"]
table_predictions = tables_cfg["predictions"]

# -------------------------------------------------------------------------
# Dev params
# -------------------------------------------------------------------------
open_time_from  = "2017-01-01 00:00:00"
open_time_to    = "2026-01-31 00:00:00"
ROLLING_WINDOW  = 240

# =============================================================================
# Database connection and query
# =============================================================================
conn = sqlite3.connect(DB_PATH)

query = f"""
	SELECT *
	FROM {table_features}
	WHERE open_time BETWEEN '{open_time_from}' AND '{open_time_to}'
	ORDER BY open_time ASC
"""
df_dev = pd.read_sql_query(query, conn)
conn.close()

# -------------------------------------------------------------------------
# Process dataframe
# -------------------------------------------------------------------------
df_dev["open_time"] = pd.to_datetime(df_dev["open_time"], utc=True)
df_dev.set_index("open_time", inplace=True)

# -------------------------------------------------------------------------
# Display
# -------------------------------------------------------------------------
print(f"\n{'='*70}")
print(f"📊 LOADED DATA FROM '{table_features}'")
print(f"{'='*70}")
print(f"Shape: {df_dev.shape}")
print(f"Time range: {df_dev.index.min()} to {df_dev.index.max()}")
print(f"\n🔍 Last 5 rows:")
print(f"{'-'*70}")
display(df_dev.tail())
print(f"{'='*70}\n")



📊 LOADED DATA FROM 'bchusdt_1m_features'
Shape: (3245657, 11)
Time range: 2019-11-28 10:00:00+00:00 to 2026-01-31 00:00:00+00:00

🔍 Last 5 rows:
----------------------------------------------------------------------


,close,trg_l_rw_240_prc_09,trg_s_rw_240_prc_01,feat_rsi_14,feat_roc_14,feat_roc_140,feat_macd_diff,feat_sma_ratio_14,feat_sma_ratio_140,feat_bb_width_14,feat_bb_width_140
open_time,,,,,,,,,,,
2026-01-30 23:56:00+00:00,553.2,0,0,53.168671,0.072359,0.072359,-0.011286,0.999600,1.000238,0.001526,0.005592
2026-01-30 23:57:00+00:00,553.2,0,0,53.168671,-0.018073,0.253715,-0.025705,0.999613,1.000220,0.001567,0.005538
2026-01-30 23:58:00+00:00,553.2,0,0,53.168671,-0.054201,0.344640,-0.035385,0.999652,1.000195,0.001604,0.005430
2026-01-30 23:59:00+00:00,553.2,0,0,53.168671,-0.072254,0.344640,-0.041508,0.999703,1.000170,0.001584,0.005317
2026-01-31 00:00:00+00:00,553.5,0,0,59.234886,0.072320,0.326264,-0.025839,1.000194,1.000690,0.001508,0.005249


In [2]:
# Target column name (from model config)
target_name = config["model"].get("target_name")
if not target_name:
    raise KeyError("target_name missing from model config")

# model

In [3]:
# =============================================================================
# parameters for logistic elimination
# =============================================================================
import statsmodels.api as sm
from IPython.display import display

p_threshold = 0.01    # p-value cutoff for keeping a variable
max_iter    = 100     # maximum elimination iterations

# =============================================================================
# prepare features list
# =============================================================================
# Expect df_dev to be available in the environment (DataFrame with feature cols and target)
features = [col for col in df_dev.columns if col.startswith("feat_")]

# =============================================================================
# CREATE df_sample BY FILTERING df_dev
# =============================================================================
# Drop rows with NA in any explanatory variable or the target
df_sample = df_dev.dropna(subset=features + [target_name])

# Exclude the most recent ROLLING_WINDOW minutes of observations
cutoff_time = df_sample.index.max() - pd.Timedelta(minutes=ROLLING_WINDOW)
df_sample   = df_sample.loc[df_sample.index < cutoff_time]

# =============================================================================
# LOGISTIC REGRESSION MODEL (iterative backward elimination)
# =============================================================================
remaining = features.copy()
removed   = []   # list of tuples (feature_name, p_value)
result    = None

for iteration in range(1, max_iter + 1):
    if not remaining:
        print("No features remaining to fit. Stopping.")
        break

    X = df_sample[remaining]
    X = sm.add_constant(X, has_constant='add')  # keep intercept
    y = df_sample[target_name]

    try:
        model = sm.Logit(y, X).fit(disp=False)
    except Exception as fit_err:
        print(f"Model fitting failed at iteration {iteration}: {fit_err}")
        # fallback: try a regularized fit to handle separation/numerical issues
        try:
            model = sm.Logit(y, X).fit_regularized(disp=False)
            print("Regularized fit succeeded as fallback.")
        except Exception as reg_err:
            print(f"Regularized fit also failed: {reg_err}. Stopping iteration.")
            break

    # p-values excluding the constant
    pvalues = model.pvalues.drop(labels=['const'], errors='ignore')

    if pvalues.empty:
        result = model
        print("No explanatory variables left after dropping constants.")
        break

    worst_var = pvalues.idxmax()
    worst_p = float(pvalues.max())

    print(f"Iteration {iteration}: {len(remaining)} features. Worst p-value = {worst_p:.6f} ({worst_var})")

    if worst_p <= p_threshold:
        result = model
        print(f"All remaining p-values <= {p_threshold}. Stopping elimination.")
        break

    # remove the worst variable and continue
    removed.append((worst_var, worst_p))
    remaining.remove(worst_var)

else:
    # reached max_iter without meeting the threshold
    print("Reached maximum iterations without satisfying p-value threshold.")
    try:
        X_final = sm.add_constant(df_sample[remaining], has_constant='add')
        result = sm.Logit(df_sample[target_name], X_final).fit(disp=False)
    except Exception as final_err:
        print("Final fit after max iterations failed:", final_err)
        result = None

# =============================================================================
# reporting results
# =============================================================================
print("\nRemoved features (in order):")
for name, pv in removed:
    print(f" - {name}: p = {pv:.6g}")

print("\nRemaining features:")
print(remaining)

if result is not None:
    try:
        display(result.summary().tables[1])
    except Exception:
        print(result.summary())
else:
    print("No final model available to display.")


Iteration 1: 8 features. Worst p-value = 0.214695 (feat_macd_diff)
Iteration 2: 7 features. Worst p-value = 0.000017 (feat_sma_ratio_140)
All remaining p-values <= 0.01. Stopping elimination.

Removed features (in order):
 - feat_macd_diff: p = 0.214695

Remaining features:
['feat_rsi_14', 'feat_roc_14', 'feat_roc_140', 'feat_sma_ratio_14', 'feat_sma_ratio_140', 'feat_bb_width_14', 'feat_bb_width_140']


,coef,std err,z,P>|z|,[0.025,0.975]
const,8.5035,1.215,6.998,0.000,6.122,10.885
feat_rsi_14,0.0064,0.000,24.271,0.000,0.006,0.007
feat_roc_14,-0.0512,0.006,-8.183,0.000,-0.063,-0.039
feat_roc_140,0.0094,0.002,4.581,0.000,0.005,0.013
feat_sma_ratio_14,-13.6024,1.128,-12.064,0.000,-15.812,-11.392
feat_sma_ratio_140,1.7650,0.411,4.297,0.000,0.960,2.570
feat_bb_width_14,61.3756,0.327,187.834,0.000,60.735,62.016
feat_bb_width_140,18.0859,0.104,173.284,0.000,17.881,18.291


# exports

In [4]:
import json
from pathlib import Path

# =============================================================================
# export only remaining feature names to JSON
# =============================================================================
output_path = Path("features.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(remaining, f, ensure_ascii=False, indent=4)

print(f"\n✅ Remaining feature nevek elmentve ide: {output_path.resolve()}")


# =============================================================================
# Slim save – adatmentes mentés, hogy a fájl kicsi maradjon
# =============================================================================

import os
import pickle

# Modellmentés célmappa és útvonal
model_path = "model.pkl"

# Előmelegítés: néhány statisztika cache-elése, hogy remove_data után is elérhető legyen
try:
    _ = model.summary()
except Exception:
    pass

# Elsődleges módszer: hivatalos statsmodels save() adatmentéssel
try:
    model.save(model_path, remove_data=True)
    print(f"✅ Slim model saved to: {model_path} (via save(remove_data=True))")

# Fallback: ha a save() remove_data paraméter nem támogatott
except Exception:
    print("⚠️ .save(remove_data=True) nem támogatott, fallback mentés indul...")
    try:
        model.remove_data()
    except Exception:
        pass
    with open(model_path, "wb") as f:
        pickle.dump(model, f)
    print(f"✅ Slim model saved to: {model_path} (via pickle fallback)")



✅ Remaining feature nevek elmentve ide: D:\repos\chronoquant\model_dev\lg_l_rw240_p90_base_sm\features.json
✅ Slim model saved to: model.pkl (via save(remove_data=True))
